In [1]:
from adaptation.misc import NameAnonymizer
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry
from controllers.retrievers.retriever import Retriever 
from services.similarity_engine import SimilarityEngine
from schemas.similarity import SearchMethod
from core.settings import get_settings
from tqdm import tqdm
import pandas as pd
import numpy as np
import time
import os


c:\Users\malos\Documents\GitHub\JustShare\server\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} spacy={'es': 'es_core_news_sm'} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/vectors.bin'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/lstm_mean_cosine'} models=[<ModelType.SBERT: 'sbert'>, <ModelType.SPACY: 'spacy'>] allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000 faiss_data_dir='./faiss_data' adaptation_data_dir='./adaptation/data' localization_dir='./adaptation/localization'


In [3]:
adaptation_dir = "./adaptation"

data_dir = os.path.join(adaptation_dir, "data")

processed_dir = os.path.join(data_dir, "processed")

database_dir = "./faiss_data"

name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")


In [4]:
name_anonymizer = NameAnonymizer(
	names_path=spanish_names_path,
	whitelist_path=name_whitelist_path,
	replacement="[UNK]"
)


In [5]:
model_registry = ModelRegistry(languages)
model_registry.build_transformer("sbert")
model_registry.build_spacy()
model_registry.build_lstm()
model_registry.build_word2vec()
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
manager = MultilingualManager(
	encoder_factory=encoder_factory,
	calibrator_factory=calibrator_factory, 
	name_anonymizer=name_anonymizer,
	base_dir=database_dir
)


2026-07-02 05:41:34.522 | DEBUG    | services.model_registry:_create_loader:57 - Registering sbert loader for 'es'.
2026-07-02 05:41:34.522 | DEBUG    | services.model_registry:_create_loader:57 - Registering spaCy loader for 'es'.
2026-07-02 05:41:34.522 | DEBUG    | services.model_registry:_create_loader:57 - Registering siamese_lstm loader for 'es'.
2026-07-02 05:41:34.522 | DEBUG    | services.model_registry:_create_loader:57 - Registering lstm calibrator loader for 'es'.
2026-07-02 05:41:34.522 | DEBUG    | services.model_registry:_create_loader:57 - Registering word2vec loader for 'es'.
2026-07-02 05:41:34.522 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-07-02 05:41:37.677 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'
2026-07-02 05:41:37.678 | DEBUG    | services.lazy_loader:model:16 - Loading word2vec for 'es'...
2026-07-02 05:41:50.991 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded word2vec for 'es'
2026-07-02 05:41:50.993 | DEBUG    | services.lazy_loader:model:16 - Loading siamese_lstm for 'es'...


Using device: cuda


2026-07-02 05:41:52.231 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded siamese_lstm for 'es'
2026-07-02 05:41:52.231 | DEBUG    | services.lazy_loader:model:16 - Loading spaCy for 'es'...
2026-07-02 05:41:52.695 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded spaCy for 'es'
2026-07-02 05:41:52.695 | DEBUG    | services.lazy_loader:model:16 - Loading lstm calibrator for 'es'...
2026-07-02 05:41:52.695 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded lstm calibrator for 'es'


In [6]:
class LeaveOneOutEvaluator:
	def __init__(self, retriever: Retriever):
		self.retriever = retriever

	def _collect_predictions(self, corpus: list[str], metadata: list[dict]):
		predictions = []

		total_time = 0
		for i, query in enumerate(corpus):
			train_corpus = corpus[:i] + corpus[i + 1:]
			train_metadata = metadata[:i] + metadata[i + 1:]

			self.retriever.fit(train_corpus)

			start = time.perf_counter()

			indices, scores, _ = self.retriever.search(
				query,
				top_k=1
			)

			total_time += time.perf_counter() - start

			if len(indices) <= 0:
				predictions.append({
					"score": 0,
					"correct": False
				})
				continue


			predicted_index = indices[0]
			
			predicted_branch = train_metadata[predicted_index]["group_index"]

			true_branch = metadata[i]["group_index"]

			predictions.append({
				"score": float(scores[0]),
				"correct": predicted_branch == true_branch
			})

		avg_time_ms = total_time / len(corpus) * 1000

		return predictions, avg_time_ms

	def evaluate(self, corpus: list[str], metadata: list[dict], thresholds: np.ndarray, best_metric: str):
		predictions, avg_time_ms = self._collect_predictions(
			corpus,
			metadata
		)
		
		threshold_results = {}

		for threshold in thresholds:
			results = self._calculate_metrics(predictions, threshold)
			threshold_results[float(threshold)] = results

		# Mejor umbral según f1
		best_threshold = max(
			threshold_results,
			key=lambda t: threshold_results[t][best_metric]
		)

		return {
			"best_threshold": best_threshold,
			"avg_time_ms": avg_time_ms,
			**threshold_results[best_threshold],
			"thresholds": threshold_results
		}

	def _calculate_metrics(self, predictions: list[dict], threshold: float):
		correct_accept = 0  # Rama correcta + encima del umbral (TP)
		wrong_accept = 0    # Rama incorrecta + encima del umbral (FP)
		correct_reject = 0  # Rama correcta + debajo del umbral (FN)
		wrong_reject = 0    # Rama incorrecta + debajo del umbral (no es TN porque todos los ejemplos pertenecen a una rama)

		total = len(predictions)

		for prediction in predictions:
			accepted = prediction["score"] >= threshold
			correct = prediction["correct"]

			if accepted and correct:
				correct_accept += 1

			elif accepted and not correct:
				wrong_accept += 1

			elif not accepted and correct:
				correct_reject += 1

			else:
				wrong_reject += 1
		
		# El porcentaje de consultas correctas
		branch_accuracy = self._safe_div(
			correct_accept,
			total
		)

		# Cuando el sistema elige una rama, cúantas veces es la correcta
		# TP = TP + FP
		precision = self._safe_div(
			correct_accept,
			correct_accept + wrong_accept
		)

		# Cuando el sistema encuentra la rama correcta, cúantas veces se supera el threshold
		# TP = TP + FN
		recall = self._safe_div(
			correct_accept,
			correct_accept + correct_reject
		)

		f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

		return {
			"branch_accuracy": branch_accuracy,
			"precision": precision,
			"recall": recall,
			"f1": f1,

			"correct_accept": correct_accept,
			"wrong_accept": wrong_accept,
			"correct_reject": correct_reject,
			"wrong_reject": wrong_reject,
		}


	def _safe_div(self, a, b):
		return a / b if b else 0
	

In [7]:
class SimilarityEvaluationRunner():
	def __init__(self, base_dir: str, retrievers: list[Retriever], min_examples: int = 2, min_groups: int = 2):
		self.base_dir = base_dir

		self.results = []
		self.situations = []

		self.retrievers = retrievers

		self.methods = set(
			retriever.encoder.name
			for retriever in self.retrievers
		)

		self.min_examples = min_examples
		self.min_groups = min_groups

	def _replace_name_placeholder(self, text: str):
		return text.replace("{{name}}", "[UNK]")

	def process_situation(self, number: int, best_metric = "f1"):
		filename = f"situation_{number}.csv"

		path = os.path.join(
			self.base_dir,
			filename
		)

		df = pd.read_csv(path, encoding="utf-8")

		groups: dict[str, list[str]] = {}
		for _, row in df.iterrows():

			group_idx = row["choice"]

			if group_idx not in groups:
				groups[group_idx] = []

			groups[group_idx].append(
				self._replace_name_placeholder(row["text_clean"])
			)

		valid_groups = {
			group_idx: texts
			for group_idx, texts in groups.items()
			if len(texts) >= self.min_examples
		}

		num_groups = len(valid_groups)

		self.situations.append({
			"situation": number,
			"num_groups": num_groups,
			"num_examples": sum(
				len(texts)
				for texts in valid_groups.values()
			),
			"evaluated": num_groups >= self.min_groups
		})

		if num_groups < self.min_groups:
			return

		corpus = []
		metadata = []

		idx = 0
		for group_idx, texts in valid_groups.items():

			for sentence_idx, text in enumerate(texts):

				corpus.append(text)

				metadata.append({
					"index": idx,
					"text": text,
					"group_index": group_idx,
					"sentence_index": sentence_idx,
					"situation": number,
				})

				idx += 1

		for retriever in self.retrievers:
			evaluator = LeaveOneOutEvaluator(
				retriever
			)

			result = evaluator.evaluate(
				corpus=corpus,
				metadata=metadata,
				thresholds=np.arange(
					0.5,
					1.0,
					0.05
				),
				best_metric=best_metric
			)

			self.results.append({
				"situation": number,
				"method": retriever.encoder.name,
				**result,
			})

	def process_situations(self, numbers: list[int], best_metric = "f1"):
		for number in tqdm(numbers):
			self.process_situation(number, best_metric)

	def get_method_results(self):
		df = (
			pd.DataFrame(self.results)
			.groupby("method", as_index=False)
			.agg({
				"best_threshold": "mean",
				"branch_accuracy": "mean",
				"precision": "mean",
				"recall": "mean",
				"f1": "mean",
				"avg_time_ms": "mean",
			})
		)

		return df
	
	def get_situation_results(self, metric: str = "f1"):
		df = pd.DataFrame(self.results)

		# Filas -> situaciones
		# Columnas -> modelos
		# Valores -> métrica
		pivot_df = (
			df.pivot(
				index="situation",
				columns="method",
				values=metric
			)
			# Ordenar las situaciones alftabéticamente
			.sort_index()
		)

		pivot_df.loc["avg"] = pivot_df.mean()
		pivot_df.loc["std"] = pivot_df.std()
		
		return pivot_df
	
	def get_situation_statistics(self):
		return pd.DataFrame(self.situations)
		

In [8]:
language = next(iter(settings.languages))

similarity_engine = SimilarityEngine(manager)

retrievers: list[Retriever] = [
	similarity_engine.get_dense_retriever(
	    method=SearchMethod.JACCARD,
		language=language
	),
	
	similarity_engine.get_dense_retriever(
		method=SearchMethod.TFIDF,
		language=language
	),
	
	similarity_engine.get_dense_retriever(
		method=SearchMethod.WORD2VEC,
		language=language
	),
	
	similarity_engine.get_dense_retriever(
	    method=SearchMethod.LSTM,
	    language=language  
	),
	
	similarity_engine.get_dense_retriever(
		method=SearchMethod.SBERT,
		language=language
	)
]


In [9]:
runner = SimilarityEvaluationRunner(
	base_dir=processed_dir,
	retrievers=retrievers
)

runner.process_situations(
	[
		1,
		2,
		3,
		4,
		5,
		6,
		7,
		8,
		9,
		10,
		11,
		12
	],
	best_metric="f1"
)


100%|██████████| 12/12 [01:35<00:00,  7.94s/it]


In [10]:
method = runner.get_method_results()
display(method)


,method,best_threshold,branch_accuracy,precision,recall,f1,avg_time_ms
0,jaccard,0.500000,0.000000,0.000000,0.000000,0.000000,0.111502
1,lstm,0.507143,0.681232,0.777845,0.880297,0.822251,57.197160
2,sbert,0.514286,0.711813,0.821720,0.890895,0.848638,8.674038
3,tfidf,0.500000,0.000000,0.000000,0.000000,0.000000,0.146860
4,word2vec,0.592857,0.505496,0.696729,0.737935,0.708760,3.176469


In [14]:
situation = runner.get_situation_results("f1")
display(situation)


method,jaccard,lstm,sbert,tfidf,word2vec
situation,,,,,
1,0.0,0.906977,0.864198,0.0,0.871795
3,0.0,0.876712,0.823529,0.0,0.806452
4,0.0,0.927536,0.911765,0.0,0.846154
5,0.0,0.526316,0.678571,0.0,0.487805
6,0.0,0.807018,0.852459,0.0,0.468085
7,0.0,0.736842,0.888889,0.0,0.536585
9,0.0,0.974359,0.921053,0.0,0.944444
avg,0.0,0.822251,0.848638,0.0,0.708760
std,0.0,0.141154,0.076213,0.0,0.187821


In [15]:
stat = runner.get_situation_statistics()
display(stat)


,situation,num_groups,num_examples,evaluated
0,1,3,48,True
1,2,1,35,False
2,3,2,41,True
3,4,2,39,True
4,5,3,41,True
5,6,2,36,True
6,7,2,37,True
7,8,1,42,False
8,9,2,42,True
9,10,1,45,False
